# Rzeznik Golf Course — explore + baseline ML

Loads data via the `golf` package, shows scorecards and hole difficulty, then
sketches a baseline model for each of the four goals.

No real rounds yet? Run `python scripts/generate_sample_data.py` first, then set
`USE_SAMPLE = True` below.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

USE_SAMPLE = False  # set True to read data/sample/ instead of real data
if USE_SAMPLE:
    from golf import schema
    schema.PLAYERS_FILE = schema.DATA_DIR / 'sample' / 'players.csv'
    schema.ROUNDS_FILE = schema.DATA_DIR / 'sample' / 'rounds.csv'
    schema.SHOTS_FILE = schema.DATA_DIR / 'sample' / 'shots.csv'

from golf import data, features
import pandas as pd, matplotlib.pyplot as plt

print('Data problems:', data.validate() or 'none')
data.holes_frame()

## Scorecards

In [ ]:
display(data.hole_scores().head(20))   # team strokes per (round, hole)
display(data.round_scores())           # team total + won (beat target)
display(data.player_contributions().head(10))  # shots / best balls per player

## 3) Hole difficulty (works with little data)

In [ ]:
hd = features.hole_difficulty()
display(hd)
if not hd.empty:
    ax = hd.sort_values('hole').plot.bar(x='hole', y='avg_to_par', legend=False)
    ax.set_ylabel('avg score to par'); ax.set_title('Hardest holes'); plt.show()

## 1) Score prediction (baseline)

Predict the team's strokes on a hole. Needs a handful of rounds before it's
meaningful — start with a simple gradient-boosted tree.

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score

df = features.score_prediction()
if len(df) >= 30:
    X = df[['hole', 'par', 'dogleg', 'blind', 'n_players']].astype(float)
    y = df['team_strokes']
    model = HistGradientBoostingRegressor(max_depth=3)
    scores = cross_val_score(model, X, y, cv=3, scoring='neg_mean_absolute_error')
    print(f'MAE: {-scores.mean():.2f} strokes')
else:
    print(f'Only {len(df)} rows — log more rounds (or use sample data) to train.')

## 4) Shot outcome (baseline classifier)

Predict a shot's outcome (one of six). With one club and no distance/lie, this is
purely categorical — expect modest signal until you have lots of shots.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

s = features.shot_outcome()
if len(s) >= 50:
    X = pd.get_dummies(s[['player_id', 'hole', 'par', 'stroke_num', 'shot_order', 'n_players']]
                       .astype({'player_id': str}))
    y = s['outcome']
    clf = RandomForestClassifier(n_estimators=200, random_state=0)
    print(f"accuracy: {cross_val_score(clf, X, y, cv=3).mean():.2f}")
else:
    print(f'Only {len(s)} shots — need more to model outcomes.')

## 2) Win probability

`features.win_probability()` gives one row per round (best-ball team result) with
`n_players` and a `prior_avg_to_par` form feature, and a `won` label (team total
beat the target score). Once you have enough rounds, train a classifier the same
way as above (target = `won`).

In [ ]:
features.win_probability()